In [ ]:
# %% [markdown]
# # Bend Distribution – Two Structures (Percentage, |bend|)
# 
# This notebook:
# - Loads two twist_bend CSV files (no header, columns: timestep, twist_deg, bend_deg)
# - Converts bend_deg → |bend_deg| (ignores sign)
# - Plots the distribution of |bend| (deg) as *percentage* histograms
# - Overlays both structures in one graph
# - Uses Arial + Illustrator‑friendly axis formatting

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

# -----------------------------------------------------------------------------
# GLOBAL STYLE
# -----------------------------------------------------------------------------

# Set font to Arial
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Arial']

# Define color scheme
COLORS = {
    'wavy': '#1F77B4',      # brighter blue
    'circular': '#FF5733',  # punchy red-orange
    'relaxed': '#FF8C00'    # vivid orange (available if needed)
}

def format_figure_for_illustrator(ax, show_right_labels=False, show_top_labels=False):
    """
    Full axis box + ticks on all sides, but labels only on left/bottom.
    """

    # 1. Full box (all spines visible)
    for side in ['top', 'right', 'left', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(1.5)
        ax.spines[side].set_color('black')

    # 2. Ticks on all 4 sides
    ax.tick_params(
        which='both',
        direction='out',
        length=4,
        width=2,
        top=True,
        right=True,
        labelsize=10
    )

    # 3. Labels only on left & bottom (unless overridden)
    ax.tick_params(
        axis='x',
        labelbottom=True,
        labeltop=show_top_labels
    )
    ax.tick_params(
        axis='y',
        labelleft=True,
        labelright=show_right_labels
    )

    # 4. Axis label font
    ax.xaxis.label.set_fontsize(10)
    ax.yaxis.label.set_fontsize(10)

    return ax

# -----------------------------------------------------------------------------
# DATA LOADING
# -----------------------------------------------------------------------------

def load_twist_bend_csv(csv_path):
    """
    Load a twist_bend.csv-like file with columns:
    timestep, twist_deg, bend_deg (no header in file).
    """
    df = pd.read_csv(
        csv_path,
        header=None,
        names=["timestep", "twist_deg", "bend_deg"],
        engine="python"
    )

    # Force numeric columns
    for col in ["timestep", "twist_deg", "bend_deg"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows with invalid twist/bend
    df = df.dropna(subset=["twist_deg", "bend_deg"])

    return df

# -----------------------------------------------------------------------------
# PLOTTING FUNCTION: BEND HISTOGRAMS (PERCENTAGE, |bend|) FOR TWO STRUCTURES
# -----------------------------------------------------------------------------

def plot_bend_hist_two_structures(
    csv_path_1,
    csv_path_2,
    label_1='wavy',
    label_2='circular',
    x_label='|Bend| (deg)',
    y_label='Percentage (%)',
    title='Distribution of |Bend| – Two Structures',
    bins=30,
    figsize=(4, 3)
):
    """
    Read two twist_bend CSV files and plot |bend| distribution for both
    structures as overlaid histograms, normalized so that each histogram
    sums to 100% on the y-axis.
    """

    # Load data
    df1 = load_twist_bend_csv(csv_path_1)
    df2 = load_twist_bend_csv(csv_path_2)

    # Take absolute value so sign is ignored
    bend1 = np.abs(df1["bend_deg"].values)
    bend2 = np.abs(df2["bend_deg"].values)

    # Create figure
    fig, ax = plt.subplots(figsize=figsize, dpi=300)

    # Common bin edges for comparability
    all_bend = np.concatenate([bend1, bend2])
    bin_edges = np.histogram_bin_edges(all_bend, bins=bins)

    # Weights so that each dataset sums to 100%
    weights1 = np.ones_like(bend1) * (100.0 / bend1.size)
    weights2 = np.ones_like(bend2) * (100.0 / bend2.size)

    # Histogram for structure 1
    ax.hist(
        bend1,
        bins=bin_edges,
        weights=weights1,
        alpha=0.5,
        label=label_1,
        color=COLORS.get(label_1, '#1F77B4'),
        edgecolor='black',
        linewidth=0.5
    )

    # Histogram for structure 2
    ax.hist(
        bend2,
        bins=bin_edges,
        weights=weights2,
        alpha=0.5,
        label=label_2,
        color=COLORS.get(label_2, '#FF5733'),
        edgecolor='black',
        linewidth=0.5
    )

    # Labels and title
    ax.set_xlabel(x_label, fontsize=10, family='Arial')
    ax.set_ylabel(y_label, fontsize=10, family='Arial')
    ax.set_title(title, fontsize=10, family='Arial', pad=10)

    # Format axes for Illustrator
    format_figure_for_illustrator(ax)

    # Legend formatting
    legend = ax.legend(loc='best', frameon=False, fontsize=10)
    for text in legend.get_texts():
        text.set_family('Arial')
        text.set_fontsize(10)

    plt.tight_layout()
    return fig, ax

# %%
# -------------------------------------------------------------------------
# EXAMPLE USAGE
# -------------------------------------------------------------------------
# Replace these with your actual filenames
csv1 = "T3IVT.csv"
csv2 = "K1IVT.csv"

fig, ax = plot_bend_hist_two_structures(
    csv1,
    csv2,
    label_1='T3IVT',
    label_2='K1IVT',
    x_label='curvature (deg)',
    y_label='Percentage (%)',
    title='Distribution of curvature',
    bins=30
)

plt.savefig("Mono_curv_abs.svg", format='svg', dpi=300, bbox_inches='tight')